# Multisensor Data Fusion with Acceleration and Position

This notebook implements a **6-state Kalman Filter** that fuses acceleration measurements from an IMU
(at 10 Hz) with position measurements from GPS (at 1 Hz).

## State Vector (Constant Acceleration Model)

$$x_k = \begin{bmatrix} x \\ y \\ \dot{x} \\ \dot{y} \\ \ddot{x} \\ \ddot{y} \end{bmatrix}$$

## Key Concepts

- **Asynchronous sensor fusion**: GPS updates arrive every 10th step; between updates, only the prediction step runs
- **Constant Acceleration Model**: Assumes acceleration is roughly constant between filter steps
- The Kalman Filter handles missing measurements gracefully by skipping the correction step

## Dependencies

- `numpy`, `scipy`, `matplotlib`, `sympy`

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sympy import Symbol, Matrix
from sympy.interactive import printing
printing.init_printing()

## State Vector
Constant Acceleration Model for Ego Motion in Plane
$$x_k= \left[ \begin{matrix} x \\ y \\ \dot x \\ \dot y \\ \ddot x \\ \ddot y \end{matrix} \right]$$

In [ ]:
x = np.array([[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]).T
print(x, x.shape)
n=x.size # States
plt.scatter(float(x[0]),float(x[1]), s=100)
plt.title('Initial Location')

In [ ]:
P = np.diag([100.0, 100.0, 10.0, 10.0, 1.0, 1.0])
print(P, P.shape)

### Time Step between Filter Steps

In [ ]:
dt = 0.1 

A = np.array([[1.0, 0.0, dt, 0.0, 1/2.0*dt**2, 0.0],
              [0.0, 1.0, 0.0, dt, 0.0, 1/2.0*dt**2],
              [0.0, 0.0, 1.0, 0.0, dt, 0.0],
              [0.0, 0.0, 0.0, 1.0, 0.0, dt],
              [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
              [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]])
print(A, A.shape)

In [ ]:
H = np.array([[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
               [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
               [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
              [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]])
print(H, H.shape)

In [ ]:
dts = Symbol('\Delta t')
Qs = Matrix([[0.5*dts**2],[0.5*dts**2],[dts],[dts],[1.0],[1.0]])
Qs*Qs.T

In [ ]:
sa = 0.001
G = np.array([[1/2.0*dt**2],
               [1/2.0*dt**2],
               [dt],
               [dt],
               [1.0],
               [1.0]])
Q = G@G.T*sa**2

print(Q, Q.shape)

In [ ]:
I = np.eye(n)
print(I, I.shape)

## Measurements
#### Assumption of  update rates:

1.Acceleration from IMU with 10Hz

2.Position from GPS with 1Hz

Which means, that every 10th of an acceleration measurement, there is a new position measurement from GPS. The Kalman Filter can perfectly handle this unsynchronous measurement incoming.


## Intialize:

*
1. Measurements
2.  Sigma for position
3.  x Position
4.  y Position


### Measurement Noise Covariance $R$

#### Intialize:
1.  ##### Noise of Acceleration Measurement $ra$
2.  ##### Noise of Position Measurement $rp$


In [ ]:
ra = 10.0**2   # Noise of Acceleration Measurement
rp = 100.0**2  # Noise of Position Measurement
R = np.array([[rp, 0.0, 0.0, 0.0],
               [0.0, rp, 0.0, 0.0],
               [0.0, 0.0, ra, 0.0],
               [0.0, 0.0, 0.0, ra]])
print(R, R.shape)

In [ ]:
m = 700

sp= 1.0 
px= 0.0
py= 0.0 

mpx = np.array(px+sp*np.random.randn(m))
mpy = np.array(py+sp*np.random.randn(m))

# Generate GPS Trigger
GPS=np.ndarray(m,dtype='bool')
GPS[0]=True
# Less new position updates
for i in range(1,m):
    if i%10==0:
        GPS[i]=True
    else:
        mpx[i]=mpx[i-1]
        mpy[i]=mpy[i-1]
        GPS[i]=False

In [ ]:
GPS

# Acceleration
### Sigma for acceleration in X and Y

In [ ]:
sa= 0.1 
ax= 0.0 
ay= 0.0 

mx = np.array(ax+sa*np.random.randn(m))
my = np.array(ay+sa*np.random.randn(m))

In [ ]:
measurements = np.vstack((mpx,mpy,mx,my))
print(measurements.shape)

In [ ]:
measurements

In [ ]:
def plot_measurements():
    fig = plt.figure(figsize=(16,9))
    plt.subplot(211)
    plt.step(range(m),mpx, label='$x$')
    plt.step(range(m),mpy, label='$y$')
    plt.ylabel(r'Position $m$')
    plt.title('Measurements')
    plt.ylim([-10, 10])
    plt.legend(loc='best',prop={'size':18})

    plt.subplot(212)
    plt.step(range(m),mx, label='$a_x$')
    plt.step(range(m),my, label='$a_y$')
    plt.ylabel(r'Acceleration $m/s^2$')
    plt.ylim([-1, 1])
    plt.legend(loc='best',prop={'size':18})

In [ ]:
plot_measurements()

# Preallocation for Plotting

In [ ]:
xt = []
yt = []
dxt= []
dyt= []
ddxt=[]
ddyt=[]
Zx = []
Zy = []
Px = []
Py = []
Pdx= []
Pdy= []
Pddx=[]
Pddy=[]
Kx = []
Ky = []
Kdx= []
Kdy= []
Kddx=[]
Kddy=[]


def savestates(x, Z, P, K):
    xt.append(float(x[0]))
    yt.append(float(x[1]))
    dxt.append(float(x[2]))
    dyt.append(float(x[3]))
    ddxt.append(float(x[4]))
    ddyt.append(float(x[5]))
    Zx.append(float(Z[0]))
    Zy.append(float(Z[1]))
    Px.append(float(P[0,0]))
    Py.append(float(P[1,1]))
    Pdx.append(float(P[2,2]))
    Pdy.append(float(P[3,3]))
    Pddx.append(float(P[4,4]))
    Pddy.append(float(P[5,5]))
    Kx.append(float(K[0,0]))
    Ky.append(float(K[1,0]))
    Kdx.append(float(K[2,0]))
    Kdy.append(float(K[3,0]))
    Kddx.append(float(K[4,0]))
    Kddy.append(float(K[5,0]))

#### The Usual Step :
1.  ##### Time Update (Prediction)
2.  ##### Project the state ahead
3.  ##### Project the error covariance ahead
4.  ##### Measurement Update (Correction)
5.  ##### if there is a GPS Measurement: Compute the Kalman Gain
6.  ##### Update the estimate via z
7.  ##### Update the error covariance
8.  ##### Save states for Plotting


In [ ]:
for kalman_filter in range(m):   
    x = A@x    
    P = A@P@A.T + Q  
    
    if GPS[kalman_filter]:        
        S = H@P@H.T + R
        K = (P@H.T) @ np.linalg.pinv(S)        
        Z = measurements[:,kalman_filter].reshape(H.shape[0],1)
        y = Z - (H@x)                            # Innovation or Residual
        x = x + (K@y)              
        P = (I - (K@H))@P

   
    
    
    savestates(x, Z, P, K)

In [ ]:
def plot_P():
    fig = plt.figure(figsize=(16,9))
    plt.subplot(211)
    plt.plot(range(len(measurements[0])),Px, label='$x$')
    plt.plot(range(len(measurements[0])),Py, label='$y$')
    plt.title('Uncertainty (Elements from Matrix $P$)')
    plt.legend(loc='best',prop={'size':22})
    plt.subplot(212)
    plt.plot(range(len(measurements[0])),Pddx, label='$\ddot x$')
    plt.plot(range(len(measurements[0])),Pddy, label='$\ddot y$')

    plt.xlabel('Filter Step')
    plt.ylabel('')
    plt.legend(loc='best',prop={'size':22})

In [ ]:
plot_P()

In [ ]:
def plot_K_ddxy():
    fig = plt.figure(figsize=(16,9))
    #plt.plot(range(len(measurements[0])),Kx, label='Kalman Gain for $x$')
    #plt.plot(range(len(measurements[0])),Ky, label='Kalman Gain for $y$')
    #plt.plot(range(len(measurements[0])),Kdx, label='Kalman Gain for $\dot x$')
    #plt.plot(range(len(measurements[0])),Kdy, label='Kalman Gain for $\dot y$')
    plt.plot(range(len(measurements[0])),Kddx, label='Kalman Gain for $\ddot x$')
    plt.plot(range(len(measurements[0])),Kddy, label='Kalman Gain for $\ddot y$')

    plt.xlabel('Filter Step')
    plt.ylabel('')
    plt.title('Kalman Gain (the lower, the more the measurement fullfill the prediction)')
    plt.legend(loc='best',prop={'size':18})

In [ ]:
plot_K_ddxy()

In [ ]:
def plot_K_dxy():
    fig = plt.figure(figsize=(16,9))
    #plt.plot(range(len(measurements[0])),Kx, label='Kalman Gain for $x$')
    #plt.plot(range(len(measurements[0])),Ky, label='Kalman Gain for $y$')
    plt.plot(range(len(measurements[0])),Kdx, label='Kalman Gain for $\dot x$')
    plt.plot(range(len(measurements[0])),Kdy, label='Kalman Gain for $\dot y$')
    #plt.plot(range(len(measurements[0])),Kddx, label='Kalman Gain for $\ddot x$')
    #plt.plot(range(len(measurements[0])),Kddy, label='Kalman Gain for $\ddot y$')

    plt.xlabel('Filter Step for dx and dy')
    plt.ylabel('')
    plt.title('Kalman Gain (the lower, the more the measurement fullfill the prediction)')
    plt.legend(loc='best',prop={'size':18})

In [ ]:
plot_K_dxy()

In [ ]:
def plot_K_xy():
    fig = plt.figure(figsize=(16,9))
    plt.plot(range(len(measurements[0])),Kx, label='Kalman Gain for $x$')
    plt.plot(range(len(measurements[0])),Ky, label='Kalman Gain for $y$')
    #plt.plot(range(len(measurements[0])),Kdx, label='Kalman Gain for $\dot x$')
    #plt.plot(range(len(measurements[0])),Kdy, label='Kalman Gain for $\dot y$')
    #plt.plot(range(len(measurements[0])),Kddx, label='Kalman Gain for $\ddot x$')
    #plt.plot(range(len(measurements[0])),Kddy, label='Kalman Gain for $\ddot y$')

    plt.xlabel('Filter Step for x and y')
    plt.ylabel('')
    plt.title('Kalman Gain (the lower, the more the measurement fullfill the prediction)')
    plt.legend(loc='best',prop={'size':18})

In [ ]:
plot_K_xy()